# SparseLevenbergMarquardtOptimizer

## Overview

`gtsam::cuda::SparseLevenbergMarquardtOptimizer` is a GPU-accelerated
Levenberg-Marquardt optimizer for **general factor graphs** — any graph whose factors
linearize to `JacobianFactor`s (Pose2/Pose3 SLAM, bundle adjustment, custom factors).
It follows the same nonlinear logic as `LevenbergMarquardtOptimizer` (damping, lambda
search, model fidelity, termination) and accelerates the expensive linear algebra on
the GPU.

**Division of work per iteration:**

- **CPU**: factor linearization (parallel, TBB) packed directly into a fixed-pattern
  sparse CSR Jacobian; state update (`retract`) and trial-error evaluation.
- **GPU**: assembly and solution of the damped normal equations
  $(J^\top J + \lambda D)\,\delta = J^\top b$ — the dominant cost on large problems.

A **symbolic plan** is computed once from the graph topology: the CSR sparsity pattern
and each factor's write offsets. All iterations reuse this fixed structure, so there is
no per-iteration symbolic work.

**Requirements.** A CUDA build of GTSAM. The cuDSS backend additionally requires cuDSS
(`GTSAM_ENABLE_CUDSS`); the PCG backend needs only cuSPARSE and works in cuDSS-free
builds. If CUDA is unavailable (or a factor type is unsupported), the optimizer
transparently falls back to CPU `LevenbergMarquardtOptimizer`
(`params.fallbackOnUnsupported`, on by default).

> For BAL-style bundle adjustment specifically, the fully GPU-resident
> `SfmLevenbergMarquardtOptimizer` (`gtsam/slam/cuda/`) is also available.


## Linear solver backends

`params.linear.backend` selects how the damped system is solved through the shared CUDA linear-solver layer:

| Backend | How it works | Character |
|---|---|---|
| `LinearSolverType::Cudss` (default) | Forms $H = J^\top J$ (sparse, on GPU) and factors it with NVIDIA cuDSS | Exact steps; supports automatic or GTSAM-supplied ordering; pays a one-time sparse *analysis* cost proportional to H |
| `LinearSolverType::Pcg` | **Matrix-free** preconditioned conjugate gradients: $H$ is never formed — the operator is applied as two SpMVs ($J\cdot p$, then $J^\top\cdot(Jp)$) | No analysis phase, no H in memory; steps are inexact (tolerance-controlled) |

**When to use which.** cuDSS is the robust direct path and gives an exact linear step.
PCG avoids normal-matrix storage and symbolic analysis, but its performance depends on
conditioning and preconditioner quality. Benchmark both on the target graph family.

**PCG details** (`params.pcg`):

- `relativeTolerance` (default `1e-6`): stop when the residual drops below this fraction
  of the gradient norm. Looser is faster but the LM trajectory diverges further from the
  exact-solver trajectory.
- `maxIterations` (default `250`): per-solve CG iteration cap. Hitting the
  cap is safe — a truncated step is still a descent direction and LM adapts.
- `warmStart` (default `true`): seed each lambda retry from the previous step, optimally
  rescaled.
- `params.pcgPreconditioner` selects block-Jacobi (production default), scalar Jacobi,
  or identity. The producer builds it from the Jacobian without forming H.


## Usage (C++)

```cpp
#include <gtsam/nonlinear/cuda/SparseLevenbergMarquardt.h>

using namespace gtsam;
using namespace gtsam::cuda;

NonlinearFactorGraph graph = ...;   // any factors (BetweenFactor, priors, SFM, ...)
Values initial = ...;

SparseLevenbergMarquardtParams params;
LevenbergMarquardtParams::SetCeresDefaults(&params);  // inherits all LM controls
params.maxIterations = 100;

// Backend selection (default: Cudss)
params.linear.backend = LinearSolverType::Pcg;
params.pcg.relativeTolerance = 1e-6;

SparseLevenbergMarquardtOptimizer optimizer(graph, initial, params);
const Values& result = optimizer.optimize();

std::cout << "final error " << optimizer.error() << "\n";
```

`SparseLevenbergMarquardtParams` derives from `LevenbergMarquardtParams`, so every
standard LM control (lambda schedule, tolerances, `iterationHook`, ...) applies
unchanged.


## Result, diagnostics, and fallback

`optimizer.result()` returns `SparseLevenbergMarquardtResult`:

- `backend`: `Cuda` or `CpuFallback` (with `fallbackReason`/`fallbackDetail` — e.g. no
  CUDA device, cuDSS not compiled for the Cudss backend, or an unsupported factor type).
- `initialError`, `finalError`, `iterations`, `lambdaAttempts`, `termination`.
- PCG counters: `pcgIterationsTotal`, `pcgSolves`, `pcgMaxIterationHits`.
- With `params.collectTiming = true`: a detailed stage breakdown in `result().timings`
  (plan construction, device setup, linearization, uploads, solve, retract, ...);
  `params.collectAttemptTrace = true` records every lambda attempt.

To *require* GPU execution instead of silently falling back (e.g. in benchmarks), set
`params.fallbackOnUnsupported = false` — construction of an unsupported configuration
then throws.

## Accuracy notes

- **cuDSS backend**: exact steps; final objectives match CPU LM to ~1e-13 relative.
- **PCG backend**: inexact steps mean LM follows a slightly different (equally valid)
  trajectory; measured final-objective differences vs CPU on the benchmark suite were
  5e-10 to 3e-3 relative at the default tolerance. At `relativeTolerance = 1e-10` the
  PCG path reproduces the CPU objective to a strict 1e-8 parity bound.

## Benchmark

`timing/cuda_sparse/timeCudaSparseLM` compares CPU LM vs both GPU backends on BAL
(Dubrovnik-16/135) and pose-graph (Pose2 w10000, Pose3 sphere) workloads with
objective-parity checks: `--gpu-solver cudss|pcg`, `--pcg-tol`, `--objective-tol`.
The checked-in Pose2 matrix records cuDSS automatic ordering, cuDSS with GTSAM
ordering, and PCG separately so ordering and backend effects are not conflated.
